# DDIM 论文详解：Denoising Diffusion Implicit Models

**论文：** *Denoising Diffusion Implicit Models*  
**作者：** Jiaming Song、Chenlin Meng、Stefano Ermon  
**发表：** ICLR 2021  
**论文地址：** https://arxiv.org/pdf/2010.02502  
**官方代码：** https://github.com/ermongroup/ddim

这篇论文是 DDPM 之后非常重要的一篇工作。它不仅提出了一种更高效的采样方法，还揭示了一个重要结论：

> **扩散模型的训练过程与采样过程并不需要严格绑定。使用相同的 DDPM 噪声预测网络，可以构造不同的反向生成过程，包括确定性的生成过程。**


## 一、先理解 DDIM 解决了什么问题

### 1. DDPM 为什么采样慢？

回顾 DDPM 的前向过程：

$$
q(x_t\mid x_{t-1})
=
\mathcal N\left(
x_t;\sqrt{\alpha_t}x_{t-1},(1-\alpha_t)I
\right)
$$

其中：

$$
\alpha_t=1-\beta_t
$$

定义：

$$
\bar\alpha_t=\prod_{i=1}^{t}\alpha_i
$$

则：

$$
\boxed{
q(x_t\mid x_0)
=
\mathcal N\left(
x_t;\sqrt{\bar\alpha_t}x_0,
(1-\bar\alpha_t)I
\right)
}
$$

所以可以直接采样：

$$
\boxed{
x_t=
\sqrt{\bar\alpha_t}x_0+
\sqrt{1-\bar\alpha_t}\epsilon
}
$$

其中：

$$
\epsilon\sim\mathcal N(0,I)
$$

训练时，网络预测：

$$
\epsilon_\theta(x_t,t)
$$

损失：

$$
\boxed{
\mathcal L_{\mathrm{simple}}
=
\mathbb E_{t,x_0,\epsilon}
\left[
\|\epsilon-\epsilon_\theta(x_t,t)\|^2
\right]
}
$$

但采样时，DDPM 通常需要执行：

$$
x_T\rightarrow x_{T-1}\rightarrow\cdots\rightarrow x_0
$$

例如 $T=1000$，就需要约 1000 次网络推理。

这使生成速度较慢。

### 2. DDIM 的关键观察

DDPM 训练时真正使用的条件分布是：

$$
q(x_t\mid x_0)
$$

而不是必须显式模拟整个前向马尔可夫链。

只要：

$$
q(x_t\mid x_0)
=
\mathcal N\left(
\sqrt{\bar\alpha_t}x_0,
(1-\bar\alpha_t)I
\right)
$$

保持不变，我们就可以继续使用相同的噪声预测训练目标。

因此作者提出：

**能否构造一种不同于 DDPM 的前向联合分布，使所有单时刻条件边缘分布保持不变，但反向采样过程更加灵活？**

答案是可以。

DDIM 构造了一族**非马尔可夫前向过程**，其中包含确定性反向采样作为特殊情况。

这带来三个重要结果：

1. 可以复用 DDPM 训练好的网络。
2. 可以跳过大量时间步，实现快速采样。
3. 可以构造确定性的生成映射，并进行图像插值和重建实验。

不过需要区分：**非马尔可夫性本身并不是跳步采样的必要条件**；DDIM 的核心优势在于构造了一个支持任意采样子序列、并与原训练目标兼容的反向更新族。

---

## 二、先统一符号：DDIM 原文与 DDPM 的记法不同

DDIM 原文使用 $\alpha_t$ 表示累计信号保留率，而我们之前学习 DDPM 时通常记为 $\bar\alpha_t$。

为避免混淆，下面统一使用：

$$
A_t=\bar\alpha_t=\prod_{i=1}^t(1-\beta_i)
$$

其中：

- $T$：训练时定义的总扩散步数。
- $t$：当前时间步。
- $\beta_t$：第 $t$ 步噪声方差。
- $A_t$：从 $x_0$ 到 $x_t$ 的累计信号保留率。
- $1-A_t$：累计噪声方差。
- $\epsilon_\theta(x_t,t)$：网络预测的噪声。
- $\sigma_t$：DDIM 构造的反向转移标准差，**不是**前向累计噪声标准差 $\sqrt{1-A_t}$。

因此：

$$
q(x_t\mid x_0)
=
\mathcal N(\sqrt{A_t}x_0,(1-A_t)I)
$$

下面正式进入论文第 3 节。

## 三、DDIM 如何构造非马尔可夫前向过程？

这是 DDIM 的第一个核心理论创新。

### 1. 从 DDPM 的联合分布开始

DDPM 的前向过程为：

$$
q(x_{1:T}\mid x_0)
=
\prod_{t=1}^Tq(x_t\mid x_{t-1})
$$

它满足马尔可夫性：

$$
q(x_t\mid x_{t-1},x_{t-2},\ldots,x_0)
=
q(x_t\mid x_{t-1})
$$

而 DDIM 的关键是：**我们不要求前向过程一定满足这种马尔可夫结构。**

我们只要求：

$$
\boxed{
q_\sigma(x_t\mid x_0)
=
\mathcal N(\sqrt{A_t}x_0,(1-A_t)I)
}
$$

与 DDPM 相同。

注意这里的条件边缘分布不变，不意味着完整的联合分布不变。

### 2. 反过来定义联合分布

DDIM 原文公式 (6)：

$$
\boxed{
q_\sigma(x_{1:T}\mid x_0)
=
q_\sigma(x_T\mid x_0)
\prod_{t=2}^T
q_\sigma(x_{t-1}\mid x_t,x_0)
}
$$

这个分解很有意思。

DDPM 是按照：

$$
x_0\rightarrow x_1\rightarrow\cdots\rightarrow x_T
$$

定义联合分布。

DDIM 则通过给定 $x_0$ 的反向条件分布：

$$
q_\sigma(x_{t-1}\mid x_t,x_0)
$$

构造联合分布。

这里的 $x_0$ 是定义训练用推断过程时的条件变量，不意味着实际生成时可以直接使用真实 $x_0$。

### 3. 推导 DDIM 的条件转移分布

我们希望设计：

$$
q_\sigma(x_{t-1}\mid x_t,x_0)
$$

使得：

$$
x_{t-1}\mid x_0
\sim
\mathcal N(\sqrt{A_{t-1}}x_0,(1-A_{t-1})I)
$$

已知：

$$
x_t=\sqrt{A_t}x_0+\sqrt{1-A_t}\epsilon
$$

其中：

$$
\epsilon\sim\mathcal N(0,I)
$$

因此，如果已知 $x_t$ 和 $x_0$，就可以计算：

$$
\boxed{
\epsilon=
\frac{x_t-\sqrt{A_t}x_0}{\sqrt{1-A_t}}
}
$$

现在构造：

$$
x_{t-1}
=
\sqrt{A_{t-1}}x_0
+
c_t\epsilon
+
\sigma_tz
$$

其中：

$$
z\sim\mathcal N(0,I)
$$

且 $z$ 与 $\epsilon$ 独立。

我们希望：

$$
\operatorname{Var}(x_{t-1}\mid x_0)
=
(1-A_{t-1})I
$$

由于两个噪声独立：

$$
\operatorname{Var}(c_t\epsilon+\sigma_tz)
=
(c_t^2+\sigma_t^2)I
$$

所以必须满足：

$$
c_t^2+\sigma_t^2=1-A_{t-1}
$$

得到：

$$
\boxed{
c_t=\sqrt{1-A_{t-1}-\sigma_t^2}
}
$$

这里取非负平方根，对应论文使用的构造。

于是：

$$
\boxed{
\begin{aligned}
x_{t-1}
={}&
\sqrt{A_{t-1}}x_0\\
&+
\sqrt{1-A_{t-1}-\sigma_t^2}
\frac{x_t-\sqrt{A_t}x_0}{\sqrt{1-A_t}}\\
&+\sigma_tz
\end{aligned}
}
$$

因此：

$$
\boxed{
\begin{aligned}
q_\sigma(x_{t-1}\mid x_t,x_0)
=
\mathcal N\Bigg(
&\sqrt{A_{t-1}}x_0\\
&+
\sqrt{1-A_{t-1}-\sigma_t^2}
\frac{x_t-\sqrt{A_t}x_0}{\sqrt{1-A_t}},
\ \sigma_t^2I
\Bigg)
\end{aligned}
}
$$

这就是论文公式 (7)。

需要满足：

$$
0\leq\sigma_t^2\leq1-A_{t-1}
$$

才能保证平方根有意义。

### 4. 为什么它保持相同的条件边缘分布？

假设：

$$
x_t\mid x_0
\sim
\mathcal N(\sqrt{A_t}x_0,(1-A_t)I)
$$

那么：

$$
\epsilon=
\frac{x_t-\sqrt{A_t}x_0}{\sqrt{1-A_t}}
\sim\mathcal N(0,I)
$$

根据刚才的构造：

$$
x_{t-1}
=
\sqrt{A_{t-1}}x_0+c_t\epsilon+\sigma_tz
$$

所以：

$$
\mathbb E[x_{t-1}\mid x_0]
=
\sqrt{A_{t-1}}x_0
$$

方差：

$$
\begin{aligned}
\operatorname{Var}(x_{t-1}\mid x_0)
&=(c_t^2+\sigma_t^2)I\\
&=(1-A_{t-1})I
\end{aligned}
$$

因此：

$$
\boxed{
q_\sigma(x_{t-1}\mid x_0)
=
\mathcal N(\sqrt{A_{t-1}}x_0,(1-A_{t-1})I)
}
$$

从 $t=T$ 开始递推，就证明所有时间步的条件边缘分布都与 DDPM 相同。

**这就是 DDIM 能够复用 DDPM 训练目标的基础。**

---

## 四、为什么非马尔可夫过程仍然可以使用 DDPM 的训练目标？

这是论文第 3.2 节和 Theorem 1 的核心。

### 1. 首先定义模型对 $x_0$ 的预测

由：

$$
x_t=\sqrt{A_t}x_0+\sqrt{1-A_t}\epsilon
$$

整理得到：

$$
x_0=
\frac{x_t-\sqrt{1-A_t}\epsilon}{\sqrt{A_t}}
$$

实际推理时不知道真实噪声，所以用网络输出替代：

$$
\boxed{
\hat x_0=
\frac{x_t-\sqrt{1-A_t}\epsilon_\theta(x_t,t)}
{\sqrt{A_t}}
}
$$

这就是论文公式 (9)。

### 2. 将 $\hat x_0$ 代入反向转移

DDIM 构造的训练用条件分布是：

$$
q_\sigma(x_{t-1}\mid x_t,x_0)
$$

但实际生成时，真实 $x_0$ 不可见。

所以定义模型：

$$
\boxed{
p_\theta(x_{t-1}\mid x_t)
=
q_\sigma(x_{t-1}\mid x_t,\hat x_0)
}
$$

也就是把条件分布中的真实 $x_0$ 换成预测值 $\hat x_0$。

因此，反向转移仍然是高斯分布：

$$
p_\theta(x_{t-1}\mid x_t)
=
\mathcal N(\mu_\theta(x_t,t),\sigma_t^2I)
$$

其中：

$$
\boxed{
\mu_\theta
=
\sqrt{A_{t-1}}\hat x_0
+
\sqrt{1-A_{t-1}-\sigma_t^2}
\epsilon_\theta(x_t,t)
}
$$

注意：这是模型定义，并不是说真实的 $q_\sigma(x_{t-1}\mid x_t)$ 一定是高斯分布。

真实反向条件分布需要对未知的 $x_0$ 积分，通常更加复杂。

### 3. 构造变分目标

与 DDPM 一样，DDIM 定义：

$$
\begin{aligned}
J_\sigma(\theta)
=
\mathbb E_{q_\sigma}
\left[
\log q_\sigma(x_{1:T}\mid x_0)
-
\log p_\theta(x_{0:T})
\right]
\end{aligned}
$$

利用：

$$
q_\sigma(x_{1:T}\mid x_0)
=
q_\sigma(x_T\mid x_0)
\prod_{t=2}^Tq_\sigma(x_{t-1}\mid x_t,x_0)
$$

以及：

$$
p_\theta(x_{0:T})
=
p(x_T)\prod_{t=1}^Tp_\theta(x_{t-1}\mid x_t)
$$

可以拆成：

$$
\boxed{
\begin{aligned}
J_\sigma
={}&
\mathbb E_{q(x_0)}
D_{\mathrm{KL}}(q_\sigma(x_T\mid x_0)\|p(x_T))\\
&+
\sum_{t=2}^T
\mathbb E_{q_\sigma(x_t,x_0)}
D_{\mathrm{KL}}
\left(
q_\sigma(x_{t-1}\mid x_t,x_0)
\|
p_\theta(x_{t-1}\mid x_t)
\right)\\
&-
\mathbb E_{q_\sigma(x_0,x_1)}
\log p_\theta(x_0\mid x_1)
\end{aligned}
}
$$

这与 DDPM 的 ELBO 分解形式非常相似。

### 4. 关键：中间 KL 为什么仍然变成噪声预测 MSE？

考虑：

$$
D_{\mathrm{KL}}
\left(
q_\sigma(x_{t-1}\mid x_t,x_0)
\|
p_\theta(x_{t-1}\mid x_t)
\right)
$$

两个高斯具有相同协方差：

$$
\sigma_t^2I
$$

因此：

$$
D_{\mathrm{KL}}
=
\frac1{2\sigma_t^2}
\|\mu_q-\mu_\theta\|^2
$$

真实条件均值：

$$
\mu_q
=
\sqrt{A_{t-1}}x_0+
c_t\epsilon
$$

其中：

$$
c_t=\sqrt{1-A_{t-1}-\sigma_t^2}
$$

模型均值：

$$
\mu_\theta
=
\sqrt{A_{t-1}}\hat x_0+
c_t\epsilon_\theta
$$

所以：

$$
\mu_q-\mu_\theta
=
\sqrt{A_{t-1}}(x_0-\hat x_0)
+
c_t(\epsilon-\epsilon_\theta)
$$

又因为：

$$
\hat x_0=
\frac{x_t-\sqrt{1-A_t}\epsilon_\theta}{\sqrt{A_t}}
$$

以及：

$$
x_0=
\frac{x_t-\sqrt{1-A_t}\epsilon}{\sqrt{A_t}}
$$

两者相减：

$$
\boxed{
x_0-\hat x_0
=
\sqrt{\frac{1-A_t}{A_t}}
(\epsilon_\theta-\epsilon)
}
$$

代回：

$$
\begin{aligned}
\mu_q-\mu_\theta
={}&
\sqrt{\frac{A_{t-1}(1-A_t)}{A_t}}
(\epsilon_\theta-\epsilon)\\
&+
c_t(\epsilon-\epsilon_\theta)
\end{aligned}
$$

合并：

$$
\boxed{
\mu_q-\mu_\theta
=
\left[
\sqrt{\frac{A_{t-1}(1-A_t)}{A_t}}
-c_t
\right]
(\epsilon_\theta-\epsilon)
}
$$

因此：

$$
\boxed{
D_{\mathrm{KL}}
=
\gamma_t
\|\epsilon_\theta-\epsilon\|^2
}
$$

其中：

$$
\boxed{
\gamma_t
=
\frac{
\left[
\sqrt{\frac{A_{t-1}(1-A_t)}{A_t}}
-
\sqrt{1-A_{t-1}-\sigma_t^2}
\right]^2
}{
2\sigma_t^2
}
}
$$

这是中间时间步的精确权重。

由于 $A_{t-1}>A_t$，在 $\sigma_t>0$ 的合法范围内，这个权重为正。

因此，每个中间 KL 都可以转化为一个带权重的噪声预测 MSE。

### 5. Theorem 1 的真正含义

论文证明，对于正的 $\sigma_t$，存在一组正权重 $\gamma_t$，使得：

$$
\boxed{
J_\sigma(\epsilon_\theta)
=
L_\gamma(\epsilon_\theta)+C
}
$$

其中：

$$
L_\gamma
=
\sum_{t=1}^T
\gamma_t
\mathbb E\|\epsilon_\theta(x_t,t)-\epsilon\|^2
$$

而普通 DDPM 使用：

$$
\boxed{
L_{\mathrm{simple}}
=
\sum_{t=1}^T
\mathbb E\|\epsilon_\theta(x_t,t)-\epsilon\|^2
}
$$

它们并不是数值完全相同。

但是，如果不同时间步的网络参数彼此独立，那么每个时间步的最优预测都是：

$$
\boxed{
\epsilon_\theta^*(x_t,t)
=
\mathbb E[\epsilon\mid x_t]
}
$$

因为正权重不会改变单独 MSE 项的最优解。

所以，不同 $\gamma_t$ 对应相同的理论最优预测器。

这就是 DDIM 能够使用 DDPM 训练网络的理论依据。

**但有一个必须注意的限制：**

实际模型通常共享参数，不同时间步之间存在参数耦合。因此，不同时间权重可能改变有限容量模型的训练结果。

论文的严格最优解论证依赖不同时间步参数不共享的条件；实际使用共享网络时，复用 DDPM 网络是一种有效的设计，并不是所有权重都严格等价。

另外，$\sigma_t=0$ 的完全确定性情形是通过极限理解的，不能直接把前面需要 $\sigma_t>0$ 的高斯 KL 公式代入零方差。

---

# 五、DDIM 最核心的采样公式

现在进入论文第 4.1 节。

我们已经得到：

$$
\hat x_0=
\frac{x_t-\sqrt{1-A_t}\epsilon_\theta(x_t,t)}
{\sqrt{A_t}}
$$

以及：

$$
p_\theta(x_{t-1}\mid x_t)
=
\mathcal N
\left(
\sqrt{A_{t-1}}\hat x_0+
\sqrt{1-A_{t-1}-\sigma_t^2}\epsilon_\theta,
\sigma_t^2I
\right)
$$

因此直接采样：

$$
\boxed{
\begin{aligned}
x_{t-1}
={}&
\underbrace{\sqrt{A_{t-1}}\hat x_0}_{\text{预测干净数据}}\\
&+
\underbrace{
\sqrt{1-A_{t-1}-\sigma_t^2}\epsilon_\theta(x_t,t)
}_{\text{保留噪声方向}}\\
&+
\underbrace{\sigma_tz}_{\text{随机噪声}}
\end{aligned}
}
$$

其中：

$$
z\sim\mathcal N(0,I)
$$

这就是论文公式 (12)。

### 1. 第一项的含义

$$
\sqrt{A_{t-1}}\hat x_0
$$

表示在下一个噪声尺度下，模型预测的干净数据成分。

### 2. 第二项的含义

$$
\sqrt{1-A_{t-1}-\sigma_t^2}\epsilon_\theta
$$

表示保留当前噪声方向的部分。

它并不是重新采样一份独立噪声，而是使用网络预测的噪声方向。

### 3. 第三项的含义

$$
\sigma_tz
$$

表示额外随机性。

因此：

$$
\boxed{
\sigma_t
}
$$

决定了 DDIM 这一族采样过程的随机程度。

当：

$$
\sigma_t=0
$$

时：

$$
\boxed{
x_{t-1}
=
\sqrt{A_{t-1}}\hat x_0+
\sqrt{1-A_{t-1}}\epsilon_\theta(x_t,t)
}
$$

这就是通常所说的**确定性 DDIM 采样**。

此时，给定初始：

$$
x_T
$$

以及固定网络，生成结果完全确定。

但这不代表 DDIM 只能生成一种样本。因为：

$$
x_T\sim\mathcal N(0,I)
$$

本身仍然是随机采样的。

不同的 $x_T$ 可以生成不同的 $x_0$。

---

## 六、DDPM 是 DDIM 采样族的一个特殊情况

这也是论文的重要结论。

对于相邻时间步，DDPM 的真实后验方差是：

$$
\boxed{
\tilde\beta_t
=
\frac{1-A_{t-1}}{1-A_t}
\left(
1-\frac{A_t}{A_{t-1}}
\right)
}
$$

如果选择：

$$
\boxed{
\sigma_t^2=\tilde\beta_t
}
$$

那么 DDIM 构造的条件分布就退化为 DDPM 对应的马尔可夫前向过程。

下面验证。

令：

$$
a_t=\frac{A_t}{A_{t-1}}
$$

则：

$$
\tilde\beta_t
=
\frac{1-A_{t-1}}{1-A_t}(1-a_t)
$$

因此：

$$
\begin{aligned}
1-A_{t-1}-\tilde\beta_t
&=
(1-A_{t-1})
\left[
1-\frac{1-a_t}{1-A_t}
\right]\\
&=
(1-A_{t-1})
\frac{A_{t-1}-A_t}{1-A_t}
\end{aligned}
$$

由于：

$$
A_t=a_tA_{t-1}
$$

所以：

$$
A_{t-1}-A_t=A_{t-1}(1-a_t)
$$

于是：

$$
\boxed{
1-A_{t-1}-\tilde\beta_t
=
\frac{A_{t-1}(1-A_{t-1})(1-a_t)}
{1-A_t}
}
$$

将这个结果代入 DDIM 条件均值，可以整理为：

$$
\boxed{
\tilde\mu_t(x_t,x_0)
=
\frac{\sqrt{A_{t-1}}(1-a_t)}{1-A_t}x_0
+
\frac{\sqrt{a_t}(1-A_{t-1})}{1-A_t}x_t
}
$$

这正是 DDPM 的后验均值。

因此：

$$
\boxed{
\sigma_t^2=\tilde\beta_t
\quad\Longrightarrow\quad
\text{DDPM 后验形式}
}
$$

需要区分 DDPM 的不同方差实现：有些实现使用 $\beta_t$ 而不是 $\tilde\beta_t$ 作为模型方差。这里恢复的是使用后验方差 $\tilde\beta_t$ 的 DDPM 形式。

---

## 七、DDIM 为什么能够跳步采样？

论文第 4.2 节进一步提出：

我们没有必要使用所有时间步：

$$
T,T-1,\ldots,1,0
$$

可以选择一个子序列：

$$
\boxed{
0=\tau_0<\tau_1<\cdots<\tau_S=T
}
$$

其中：

$$
S\ll T
$$

例如：

$$
T=1000,\qquad S=20
$$

就可以使用：

$$
1000\rightarrow950\rightarrow900\rightarrow\cdots\rightarrow0
$$

这种较稀疏的采样路径。

设当前时间为：

$$
t=\tau_i
$$

下一个时间为：

$$
s=\tau_{i-1}
$$

其中：

$$
s<t
$$

那么跳步 DDIM 的更新公式是：

$$
\boxed{
x_s
=
\sqrt{A_s}\hat x_0+
\sqrt{1-A_s-\sigma_{t\to s}^2}
\epsilon_\theta(x_t,t)
+
\sigma_{t\to s}z
}
$$

这里：

$$
\hat x_0=
\frac{x_t-\sqrt{1-A_t}\epsilon_\theta(x_t,t)}
{\sqrt{A_t}}
$$

论文使用参数：

$$
\eta\geq0
$$

控制随机性：

$$
\boxed{
\sigma_{t\to s}(\eta)
=
\eta
\sqrt{\frac{1-A_s}{1-A_t}}
\sqrt{1-\frac{A_t}{A_s}}
}
$$

这就是论文公式 (16) 的一般形式。

### 1. $\eta=0$

得到：

$$
\sigma_{t\to s}=0
$$

所以：

$$
\boxed{
x_s=
\sqrt{A_s}\hat x_0+
\sqrt{1-A_s}\epsilon_\theta(x_t,t)
}
$$

完全确定性。

### 2. $\eta=1$

得到与选定时间子序列对应的后验方差。

当使用完整相邻时间步时，它恢复 DDPM 后验方差形式。

### 3. 为什么跳步不需要重新训练？

因为网络训练时学习的是：

$$
\epsilon_\theta(x_t,t)
$$

而任意时间 $t$ 的条件边缘分布仍然是：

$$
q(x_t\mid x_0)
=
\mathcal N(\sqrt{A_t}x_0,(1-A_t)I)
$$

因此，模型在时间 $t$ 的输入分布没有改变。

变化的只是从 $t$ 到 $s$ 的生成更新方式。

所以：

$$
\boxed{
\text{同一个网络}
+
\text{不同时间子序列}
=
\text{不同采样速度}
}
$$

当然，跳步越大，数值离散化误差通常也越明显。

---

# 八、DDIM 与 Probability Flow ODE 的关系

这是我认为整篇论文中最值得深入理解的部分，尤其与你之前研究的 DPM-Solver、Flow Matching 直接相关。

论文第 4.3 节证明，确定性 DDIM 可以看作某个 ODE 的数值离散化。

## 1. 从确定性 DDIM 更新开始

令：

$$
\sigma_{t\to s}=0
$$

则：

$$
x_s=
\sqrt{A_s}\hat x_0+
\sqrt{1-A_s}\epsilon_\theta(x_t,t)
$$

代入：

$$
\hat x_0=
\frac{x_t-\sqrt{1-A_t}\epsilon_\theta(x_t,t)}
{\sqrt{A_t}}
$$

得到：

$$
\begin{aligned}
x_s
={}&
\sqrt{\frac{A_s}{A_t}}x_t\\
&-
\sqrt{\frac{A_s(1-A_t)}{A_t}}
\epsilon_\theta(x_t,t)\\
&+
\sqrt{1-A_s}\epsilon_\theta(x_t,t)
\end{aligned}
$$

合并：

$$
\boxed{
x_s
=
\sqrt{\frac{A_s}{A_t}}x_t
+
\left[
\sqrt{1-A_s}
-
\sqrt{\frac{A_s(1-A_t)}{A_t}}
\right]
\epsilon_\theta(x_t,t)
}
$$

两边除以：

$$
\sqrt{A_s}
$$

得到：

$$
\boxed{
\frac{x_s}{\sqrt{A_s}}
=
\frac{x_t}{\sqrt{A_t}}
+
\left[
\sqrt{\frac{1-A_s}{A_s}}
-
\sqrt{\frac{1-A_t}{A_t}}
\right]
\epsilon_\theta(x_t,t)
}
$$

这就是论文公式 (13)。

## 2. 引入新的变量

定义：

$$
\boxed{
y_t=\frac{x_t}{\sqrt{A_t}}
}
$$

再定义：

$$
\boxed{
r_t=\sqrt{\frac{1-A_t}{A_t}}
}
$$

注意：

$$
r_t
$$

是噪声与信号的标准差之比，不是 DDIM 反向更新的随机噪声标准差 $\sigma_t$。

于是刚才的公式变成：

$$
\boxed{
y_s=y_t+(r_s-r_t)\epsilon_\theta(x_t,t)
}
$$

因为：

$$
x_t=\sqrt{A_t}y_t
$$

并且：

$$
A_t=\frac1{1+r_t^2}
$$

所以：

$$
x_t=\frac{y_t}{\sqrt{1+r_t^2}}
$$

因此：

$$
\boxed{
y_s
=
y_t+
(r_s-r_t)
\epsilon_\theta
\left(
\frac{y_t}{\sqrt{1+r_t^2}},t
\right)
}
$$

观察这个公式：

$$
y_s=y_t+\Delta r\cdot F(y_t,r_t)
$$

它就是标准的 Euler 积分形式。

所以当时间步趋于无穷小时：

$$
\boxed{
\frac{dy}{dr}
=
\epsilon_\theta
\left(
\frac{y}{\sqrt{1+r^2}},t(r)
\right)
}
$$

这就是 DDIM 对应的连续时间 ODE。

### 3. 为什么这个 ODE 与 Probability Flow ODE 等价？

我们之前推导 VP-SDE：

$$
dX_t=
-\frac12\beta(t)X_tdt+
\sqrt{\beta(t)}dW_t
$$

其 Probability Flow ODE 为：

$$
\boxed{
\frac{dx}{dt}
=
-\frac12\beta(t)x
-
\frac12\beta(t)s_\theta(x,t)
}
$$

对于 VP-SDE：

$$
A(t)=e^{-B(t)}
$$

其中：

$$
B(t)=\int_0^t\beta(u)du
$$

因此：

$$
\frac{dA}{dt}=-\beta(t)A(t)
$$

定义：

$$
\sigma(t)=\sqrt{1-A(t)}
$$

网络噪声预测与 Score 的关系为：

$$
\boxed{
s_\theta(x,t)
=
-\frac{\epsilon_\theta(x,t)}{\sigma(t)}
}
$$

代入 Probability Flow ODE：

$$
\frac{dx}{dt}
=
-\frac12\beta x+
\frac{\beta}{2\sigma}\epsilon_\theta
$$

现在定义：

$$
y=\frac{x}{\sqrt A}
$$

对时间求导：

$$
\frac{dy}{dt}
=
\frac1{\sqrt A}\frac{dx}{dt}
-
\frac{x}{2A^{3/2}}\frac{dA}{dt}
$$

由于：

$$
\frac{dA}{dt}=-\beta A
$$

所以：

$$
-\frac{x}{2A^{3/2}}\frac{dA}{dt}
=
\frac{\beta x}{2\sqrt A}
$$

因此：

$$
\begin{aligned}
\frac{dy}{dt}
&=
\frac1{\sqrt A}
\left[
-\frac12\beta x+
\frac{\beta}{2\sigma}\epsilon_\theta
\right]
+
\frac{\beta x}{2\sqrt A}\\
&=
\frac{\beta}{2\sqrt A\sigma}\epsilon_\theta
\end{aligned}
$$

另一方面：

$$
r=\frac{\sigma}{\sqrt A}
=
\sqrt{\frac{1-A}{A}}
$$

所以：

$$
r^2=\frac1A-1
$$

对时间求导：

$$
2r\frac{dr}{dt}
=
-\frac1{A^2}\frac{dA}{dt}
$$

代入：

$$
\frac{dA}{dt}=-\beta A
$$

得到：

$$
2r\frac{dr}{dt}
=
\frac{\beta}{A}
$$

所以：

$$
\frac{dr}{dt}
=
\frac{\beta}{2Ar}
$$

由于：

$$
Ar=\sqrt A\sigma
$$

因此：

$$
\boxed{
\frac{dr}{dt}
=
\frac{\beta}{2\sqrt A\sigma}
}
$$

最终：

$$
\frac{dy}{dr}
=
\frac{dy/dt}{dr/dt}
$$

得到：

$$
\boxed{
\frac{dy}{dr}
=
\epsilon_\theta(x,t)
}
$$

而：

$$
x=\frac{y}{\sqrt{1+r^2}}
$$

所以：

$$
\boxed{
\frac{dy}{dr}
=
\epsilon_\theta
\left(
\frac{y}{\sqrt{1+r^2}},t(r)
\right)
}
$$

这与 DDIM 推导出的 ODE 完全一致。

因此：

$$
\boxed{
\text{DDIM 的连续时间极限}
=
\text{VP-SDE Probability Flow ODE}
}
$$

这里说的是**连续时间动力学等价**。

在有限步数下，DDIM 使用 $r$ 作为积分变量，而直接对 VP Probability Flow ODE 使用时间 $t$ 做 Euler 积分，得到的离散更新通常不同。

因此，不能简单认为：

$$
\text{DDIM 单步更新}
=
\text{直接 Euler 积分 VP-ODE}
$$

两者只有在适当连续极限下才一致。

---

# 九、DDIM 与 DDPM、DPM-Solver 的关系

现在我们可以把三者联系起来。

| 方法 | 数学基础 | 采样随机性 | 主要特点 |
|---|---|---|---|
| DDPM | 离散反向扩散过程 | 通常有 | 逐步去噪 |
| DDIM | 非马尔可夫过程及其隐式生成模型 | 可为零 | 确定性、支持跳步 |
| DPM-Solver | 扩散 Probability Flow ODE | 无 | 利用半线性结构进行高阶求解 |

从 ODE 角度看：

DDIM 相当于在特定变量变换下，使用一阶方法进行积分。

DPM-Solver 则进一步利用扩散 ODE 的结构，构造高阶求解器。

这也是为什么 DDIM 是理解 DPM-Solver 的重要基础。

## 十、论文实验部分的结论

作者在 CIFAR-10 和 CelebA 上进行了采样实验，比较不同采样步数和随机性参数 $\eta$。

例如 CIFAR-10 的 FID：

| 采样步数 | DDIM，$\eta=0$ | $\eta=1$ |
|---|---:|---:|
| 10 | 13.36 | 41.07 |
| 20 | 6.84 | 18.36 |
| 50 | 4.67 | 8.01 |
| 100 | 4.16 | 5.78 |
| 1000 | 4.04 | 4.73 |

这些是论文特定模型和方差配置下的结果，不能解释为 DDIM 在所有设置下都优于 DDPM。

论文还展示了三个有意思的性质：

**第一，少步生成。** DDIM 在较少网络调用次数下仍能保持较好的样本质量。

**第二，生成一致性。** 固定同一个初始噪声 $x_T$，改变 DDIM 的采样步数，生成样本的高层语义往往相似。

**第三，可逆性与重建。** 因为 DDIM 对应确定性 ODE，可以从数据向噪声方向进行近似积分，再从噪声向数据方向重建。但有限步离散化和网络误差会带来重建误差。[arXiv](https://arxiv.org/html/2010.02502?utm_source=chatgpt.com)

---

# 十一、这篇论文需要真正掌握的五个结论

### 1. DDIM 改变的是联合分布，不是训练时的单时刻条件边缘分布

$$
\boxed{
q_\sigma(x_{1:T}\mid x_0)
\neq
q_{\mathrm{DDPM}}(x_{1:T}\mid x_0)
}
$$

但：

$$
\boxed{
q_\sigma(x_t\mid x_0)
=
q_{\mathrm{DDPM}}(x_t\mid x_0)
}
$$

这是复用 DDPM 训练目标的根本原因。

### 2. DDIM 的确定性来自反向采样方差为零

$$
\boxed{
\sigma_t=0
}
$$

并不意味着初始噪声 $x_T$ 不随机。

### 3. DDIM 可以跳步采样

$$
\boxed{
x_t\rightarrow x_s,\qquad s<t
}
$$

不需要执行所有中间时间步。

### 4. DDIM 与 Probability Flow ODE 有直接联系

通过变量变换：

$$
\boxed{
y=\frac{x}{\sqrt A},
\qquad
r=\sqrt{\frac{1-A}{A}}
}
$$

可以将 DDIM 写成：

$$
\boxed{
\frac{dy}{dr}
=
\epsilon_\theta
\left(
\frac{y}{\sqrt{1+r^2}},t(r)
\right)
}
$$

这就是理解后续高阶扩散 ODE 求解器的关键。

### 5. DDIM 的训练目标并非对所有模型都严格等价于 DDPM

Theorem 1 证明的是带权噪声预测目标的结构等价；使用统一的无权重 MSE 作为替代目标，还需要考虑不同时间步参数共享、模型容量和优化误差。

---

## 十二、与后续 Flow Matching 的联系

你之前研究 Flow Matching 时，经常讨论一个问题：

> 如果模型学习的是速度场，那么为什么可以直接用 Euler、Heun 或 DPM-Solver 进行积分？

DDIM 已经提供了一个重要线索：

**同一个连续时间生成过程，在不同变量和时间参数化下，可能对应不同的数值积分公式。**

例如 DDIM 的更新：

$$
y_s=y_t+(r_s-r_t)\epsilon_\theta(x_t,t)
$$

就是在 $r$ 空间进行 Euler 积分。

而普通 Flow Matching：

$$
\frac{dx}{dt}=v_\theta(x,t)
$$

通常直接在 $t$ 空间进行积分。

两者的区别不仅在于网络预测的是噪声还是速度，还在于：

- 状态变量如何定义；
- 时间变量如何参数化；
- 生成 ODE 是否具有可利用的解析结构；
- 数值积分器在哪个变量空间中工作。

